# Fashion MNIST — klasyfikacja, zapis modelu i augmentacja

Notebook realizuje trzy części zadania:

1. przygotowanie danych z wymaganym podziałem i trening modelu CNN,
2. zapis oraz ponowne wczytanie modelu i interfejs predykcji,
3. trening mocniejszego modelu z augmentacją danych.

> Wynik zależy od wersji TensorFlow, sprzętu i losowania. Dążymy do `accuracy > 0.94`,
> ale nie wykorzystujemy zbioru testowego do dobierania modelu. Zbiór testowy jest
> używany dopiero do końcowej, uczciwej oceny.

## 1. Import bibliotek i ustawienie losowości

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers

# Stałe ziarno zwiększa powtarzalność wyników.
SEED = 10
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Dostępne GPU:", tf.config.list_physical_devices("GPU"))

## 2. Pobranie i przygotowanie danych

In [ ]:
# Fashion MNIST zawiera 60 000 obrazów treningowych i 10 000 testowych.
# Zgodnie z treścią zadania korzystamy z tablic images i labels,
# a następnie wykonujemy wymagany train_test_split.
(images, labels), _ = tf.keras.datasets.fashion_mnist.load_data()

# Zamiana pikseli z zakresu 0–255 na liczby float32 z zakresu 0–1.
images = images.astype("float32") / 255.0
labels = labels.astype("int32")

class_names = [
    "T-shirt/top", "Spodnie", "Sweter", "Sukienka", "Płaszcz",
    "Sandał", "Koszula", "But sportowy", "Torba", "Botek"
]

# Dokładnie taki podział został wymagany w zadaniu.
# stratify zachowuje proporcje dziesięciu klas w obu częściach.
X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.1,
    random_state=10,
    stratify=labels
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

### Osobny zbiór walidacyjny

Zbioru testowego nie używamy do wyboru epoki ani strojenia parametrów. Z danych
treningowych wydzielamy więc walidację. Dopiero gotowy model oceniamy na `X_test`.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train
)

print("Trening:  ", X_fit.shape)
print("Walidacja:", X_val.shape)
print("Test:      ", X_test.shape)

In [ ]:
# Wyświetlenie przykładowych danych wejściowych.
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, image, label in zip(axes.ravel(), X_fit[:10], y_fit[:10]):
    axis.imshow(image, cmap="gray")
    axis.set_title(class_names[int(label)])
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Funkcja budująca model CNN

In [ ]:
def build_cnn(use_augmentation=False):
    # Augmentacja działa wyłącznie podczas treningu.
    augmentation = tf.keras.Sequential(
        [
            layers.RandomTranslation(0.08, 0.08, fill_mode="constant", seed=SEED),
            layers.RandomRotation(0.04, fill_mode="constant", seed=SEED),
            layers.RandomZoom(
                height_factor=(-0.08, 0.08),
                width_factor=(-0.08, 0.08),
                fill_mode="constant",
                seed=SEED,
            ),
        ],
        name="augmentation",
    )

    inputs = layers.Input(shape=(28, 28), name="image")
    x = layers.Reshape((28, 28, 1))(inputs)

    if use_augmentation:
        x = augmentation(x)

    # Pierwszy blok wydobywa proste cechy, np. krawędzie.
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.20)(x)

    # Drugi blok uczy się bardziej złożonych fragmentów ubrań.
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    # Trzeci blok zwiększa pojemność modelu.
    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.35)(x)

    outputs = layers.Dense(10, activation="softmax", name="probabilities")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 4. Model bazowy — bez augmentacji

In [ ]:
baseline_model = build_cnn(use_augmentation=False)
baseline_model.summary()

In [ ]:
# Callbacki zatrzymują trening, gdy walidacja przestaje się poprawiać,
# zmniejszają learning rate i zachowują najlepszą wersję modelu.
baseline_path = Path("fashion_mnist_baseline.keras")

baseline_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        baseline_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1,
    ),
]

baseline_history = baseline_model.fit(
    X_fit,
    y_fit,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=128,
    callbacks=baseline_callbacks,
    verbose=1,
)

In [ ]:
def plot_history(history, title):
    # Wykres ułatwia ocenę, czy model nadal się uczy i czy się przeucza.
    history_df = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    history_df[["accuracy", "val_accuracy"]].plot(ax=axes[0])
    axes[0].set_title(f"{title} — accuracy")
    axes[0].set_xlabel("Epoka")
    axes[0].set_ylim(0.75, 1.0)
    axes[0].grid(True)

    history_df[["loss", "val_loss"]].plot(ax=axes[1])
    axes[1].set_title(f"{title} — loss")
    axes[1].set_xlabel("Epoka")
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


plot_history(baseline_history, "Model bazowy")

In [ ]:
# Wczytujemy najlepszą epokę zapisaną przez ModelCheckpoint.
baseline_model = tf.keras.models.load_model(baseline_path)
baseline_loss, baseline_accuracy = baseline_model.evaluate(
    X_test, y_test, verbose=0
)
print(f"Model bazowy — test loss: {baseline_loss:.4f}")
print(f"Model bazowy — test accuracy: {baseline_accuracy:.4f}")

## 5. Interfejs predykcji i wizualizacja

In [ ]:
def predict_and_show(value, model=baseline_model, images=X_test, labels=y_test):
    """Przyjmuje indeks obrazu albo tablicę 28×28 i pokazuje predykcję.

    Przykłady:
        predict_and_show(2)
        predict_and_show(X_test[2])
    """
    if np.isscalar(value):
        index = int(value)
        if index < 0 or index >= len(images):
            raise IndexError(f"Indeks musi należeć do zakresu 0–{len(images) - 1}.")
        image = images[index]
        real_class = int(labels[index])
    else:
        image = np.asarray(value, dtype="float32")
        if image.shape != (28, 28):
            raise ValueError("Obraz musi mieć kształt (28, 28).")
        real_class = None

    probabilities = model.predict(image[np.newaxis, ...], verbose=0)[0]
    predicted_class = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_class])

    plt.figure(figsize=(4, 4))
    plt.imshow(image, cmap="gray")
    title = (
        f"Model: {class_names[predicted_class]} "
        f"({confidence * 100:.2f}%)"
    )
    if real_class is not None:
        title += f"\nRzeczywista: {class_names[real_class]}"
    plt.title(title, color="green" if real_class == predicted_class else "red")
    plt.axis("off")
    plt.show()

    result = {
        "numer_klasy": predicted_class,
        "nazwa_klasy": class_names[predicted_class],
        "pewnosc": confidence,
        "prawdopodobienstwa": probabilities,
    }
    print(result)
    return result


# Przykładowe użycie interfejsu.
prediction = predict_and_show(2)

### Opcjonalny suwak w Jupyter Notebook

In [ ]:
# Jeżeli ipywidgets jest zainstalowane, pojawi się interaktywny suwak.
try:
    import ipywidgets as widgets
    from IPython.display import display

    index_slider = widgets.IntSlider(
        value=2,
        min=0,
        max=len(X_test) - 1,
        step=1,
        description="Indeks:",
        continuous_update=False,
    )
    widget_output = widgets.interactive_output(
        lambda index: predict_and_show(index, model=baseline_model),
        {"index": index_slider},
    )
    display(index_slider, widget_output)
except ImportError:
    print("Brak ipywidgets — nadal można używać predict_and_show(indeks).")

## 6. Model z augmentacją danych

In [ ]:
# Podgląd transformacji pozwala ocenić, czy obrazy nadal wyglądają realistycznie.
preview_augmentation = tf.keras.Sequential(
    [
        layers.RandomTranslation(0.08, 0.08, fill_mode="constant", seed=SEED),
        layers.RandomRotation(0.04, fill_mode="constant", seed=SEED),
        layers.RandomZoom((-0.08, 0.08), (-0.08, 0.08),
                          fill_mode="constant", seed=SEED),
    ]
)

sample = X_fit[0][np.newaxis, ..., np.newaxis]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis in axes.ravel():
    augmented = preview_augmentation(sample, training=True)[0, ..., 0]
    axis.imshow(augmented, cmap="gray")
    axis.axis("off")
plt.suptitle("Dziesięć wariantów tego samego obrazu")
plt.tight_layout()
plt.show()

In [ ]:
augmented_model = build_cnn(use_augmentation=True)
augmented_path = Path("fashion_mnist_augmented.keras")

augmented_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        augmented_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=9,
        restore_best_weights=True,
        verbose=1,
    ),
]

augmented_history = augmented_model.fit(
    X_fit,
    y_fit,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    callbacks=augmented_callbacks,
    verbose=1,
)

In [ ]:
plot_history(augmented_history, "Model z augmentacją")

augmented_model = tf.keras.models.load_model(augmented_path)
augmented_loss, augmented_accuracy = augmented_model.evaluate(
    X_test, y_test, verbose=0
)

print(f"Model bazowy — test accuracy:     {baseline_accuracy:.4f}")
print(f"Model z augmentacją — accuracy:   {augmented_accuracy:.4f}")
print(f"Różnica:                          {augmented_accuracy - baseline_accuracy:+.4f}")

## 7. Wybór i zapis najlepszego modelu

In [ ]:
# Wybieramy model wyłącznie na podstawie walidacji, nie testu.
baseline_best_val = max(baseline_history.history["val_accuracy"])
augmented_best_val = max(augmented_history.history["val_accuracy"])

if augmented_best_val >= baseline_best_val:
    final_model = augmented_model
    selected_name = "model z augmentacją"
else:
    final_model = baseline_model
    selected_name = "model bazowy"

final_model_path = Path("fashion_mnist_final.keras")
final_model.save(final_model_path)

print("Wybrano:", selected_name)
print("Zapisano model:", final_model_path.resolve())

In [ ]:
# Sprawdzenie, czy zapisany plik rzeczywiście można odtworzyć.
loaded_model = tf.keras.models.load_model("fashion_mnist_final.keras")
loaded_loss, loaded_accuracy = loaded_model.evaluate(X_test, y_test, verbose=0)

print(f"Wczytany model — test loss: {loaded_loss:.4f}")
print(f"Wczytany model — test accuracy: {loaded_accuracy:.4f}")

# Interfejs korzystający z modelu odczytanego z dysku.
predict_and_show(2, model=loaded_model)

## 8. Szczegółowa ocena końcowa

In [ ]:
final_probabilities = loaded_model.predict(X_test, batch_size=256, verbose=0)
final_predictions = np.argmax(final_probabilities, axis=1)

print(classification_report(
    y_test,
    final_predictions,
    target_names=class_names,
    digits=4,
))

fig, axis = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_predictions,
    display_labels=class_names,
    cmap="Blues",
    xticks_rotation=45,
    ax=axis,
    colorbar=False,
)
plt.title("Macierz pomyłek — Fashion MNIST")
plt.tight_layout()
plt.show()

## Wnioski

- Model jest oceniany na odłożonych 10% danych zgodnie z podziałem wymaganym w zadaniu.
- `ModelCheckpoint` zapisuje najlepszą epokę, a format `.keras` przechowuje cały model.
- Funkcja `predict_and_show()` jest prostym interfejsem przyjmującym indeks lub obraz.
- Augmentacja generuje realistycznie przesunięte, obrócone i powiększone warianty obrazów.
- Wyniku około 0.97 nie należy gwarantować. Trzeba podać faktycznie uzyskany wynik;
  na Fashion MNIST klasy `T-shirt/top`, `Koszula`, `Sweter` i `Płaszcz` są do siebie podobne.

Źródła:

- [TensorFlow — zapisywanie i wczytywanie modeli](https://www.tensorflow.org/tutorials/keras/save_and_load)
- [TensorFlow — augmentacja danych](https://www.tensorflow.org/tutorials/images/data_augmentation)
- [TensorFlow — ModelCheckpoint](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ModelCheckpoint)